In [110]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from abc import ABC, abstractmethod
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch
import torch.optim as optim

In [29]:
dataset = pd.read_csv("geotechnical_settlement_dataset.csv")

In [4]:
dataset.head()

,Depth (m),Soil Type,Unit Weight γ (kN/m³),Moisture Content (%),SPT N-value,Foundation Width B (m),Applied Load q (kPa),Elastic Modulus E (MPa),Poisson Ratio ν,Influence Factor Is,Settlement S (mm),Bearing Capacity (kPa)
0,7.13,Sand,17.11,23.17,22,1.15,256.88,66.93,0.286,1.053,5.00,214.97
1,5.59,Silt,19.25,21.77,16,1.12,241.13,29.95,0.341,1.062,8.69,161.80
2,3.44,Sand,21.24,5.90,28,2.77,193.38,85.70,0.318,0.979,5.98,277.31
3,8.23,Sand,20.39,16.94,17,1.12,289.01,48.53,0.256,0.967,5.91,173.28
4,7.00,Clay,20.84,18.31,6,1.18,100.12,9.55,0.403,1.182,12.62,54.65


In [167]:

class BaseModel(ABC):
    def __init__(self, name):
        if not name:
            raise ValueError("Model name is required")
        self.name = name
    @abstractmethod
    def clean_data(self):
        pass
        
    # @abstractmethod
    # def train(self, X, y):
    #     pass


class Settlement(BaseModel):
    
    def __init__(self,dataset_path):
        if not dataset_path:
            raise ValueError("Dataset is required")
        print(dataset_path)
        try:
            self.dataset = pd.read_csv(dataset_path)  # will cause error
        except:
            print("data not found")

    @staticmethod
    def fill_null(dataset):
        for col in list(dataset.columns):
    
            if dataset[col].dtype == 'O':
                dataset[col] = dataset[col].fillna("Unknown")
                print(col)
            else:
            # check for skewness
                if int(dataset[col].skew())>0.5:
                
                    dataset[col] = dataset[col].fillna(dataset[col].median())
                else:
                    dataset[col] = dataset[col].fillna(dataset[col].mean())
            
            
    

    def clean_data(self):
        if self.dataset.isnull().sum().sum() == 0:
            print("data is clean")
        else:
            Settlement.fill_null(self.dataset)


    def one_hot_encoder(self,column_name):
        # input row that needs encoding
        encoder = OneHotEncoder(sparse_output=False)
        encoded = encoder.fit_transform(self.dataset[[column_name]])
        encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out())
        self.dataset = pd.concat([self.dataset.drop(columns=[f"{column_name}"]), encoded_df], axis=1)
        print(self.dataset.head(1))


    def split_and_normalize(self,target):
        X = self.dataset.drop(columns = [f"{target}","Bearing Capacity (kPa)"])
        y = self.dataset[f"{target}"]

        X_train, X_temp, y_train, y_temp = train_test_split(
            X, y, test_size=0.3, random_state=42)
        
        X_val, X_test, y_val, y_test = train_test_split(
           X_temp, y_temp, test_size=0.5, random_state=42)

        scaler = MinMaxScaler()

        X_train = scaler.fit_transform(X_train)
        # learn + scale
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)  
        
        return (X_train,X_test,X_val,y_train,y_test,y_val)
                              

                            




        

class Model(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)





        
        
    
        

In [168]:
model = Settlement("geotechnical_settlement_dataset.csv")

geotechnical_settlement_dataset.csv


In [169]:
model.one_hot_encoder("Soil Type")

   Depth (m)  Unit Weight γ (kN/m³)  Moisture Content (%)  SPT N-value  \
0       7.13                  17.11                 23.17           22   

   Foundation Width B (m)  Applied Load q (kPa)  Elastic Modulus E (MPa)  \
0                    1.15                256.88                    66.93   

   Poisson Ratio ν  Influence Factor Is  Settlement S (mm)  \
0            0.286                1.053                5.0   

   Bearing Capacity (kPa)  Soil Type_Clay  Soil Type_Sand  Soil Type_Silt  
0                  214.97             0.0             1.0             0.0  


In [171]:

X_train,X_test,X_val,y_train,y_test,y_val = model.split_and_normalize("Settlement S (mm)")

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)




In [156]:
new_model = Model(input_size=X_train.shape[1])

In [120]:
print(new_model.net[2].weight.shape)

torch.Size([16, 32])


In [178]:
criterion = nn.MSELoss()
optimizer = optim.Adam(new_model.parameters(), lr=0.001)

In [180]:
for epoch in range(300):
    new_model.train()
    
    y_pred = new_model(X_train)
    loss = criterion(y_pred, y_train)
    
    optimizer.zero_grad()
    #set gradient to zero so we can learn from new gradient that will actually reduce the loss function
    loss.backward()
    # after calculating our loss founction, calculates how much each weight caused the error
    optimizer.step()
    #Updates weights to reduce error
    print(y_val[0])
    new_model.eval()
    with torch.no_grad():
        val_pred = new_model(X_val)
        val_loss = criterion(val_pred, y_val)

    print(f"Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")
    
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, Prediction for y_pred → {val_pred[0].detach().numpy()}")

tensor([3.1700])
Train Loss: 1.2938, Val Loss: 3.7890
Epoch 1: Loss = 1.2938, Prediction for y_pred → [2.1965098]
tensor([3.1700])
Train Loss: 1.2922, Val Loss: 3.8017
Epoch 2: Loss = 1.2922, Prediction for y_pred → [2.1951213]
tensor([3.1700])
Train Loss: 1.2905, Val Loss: 3.7971
Epoch 3: Loss = 1.2905, Prediction for y_pred → [2.1933622]
tensor([3.1700])
Train Loss: 1.2890, Val Loss: 3.7866
Epoch 4: Loss = 1.2890, Prediction for y_pred → [2.1915731]
tensor([3.1700])
Train Loss: 1.2871, Val Loss: 3.7851
Epoch 5: Loss = 1.2871, Prediction for y_pred → [2.190099]
tensor([3.1700])
Train Loss: 1.2854, Val Loss: 3.7833
Epoch 6: Loss = 1.2854, Prediction for y_pred → [2.1888757]
tensor([3.1700])
Train Loss: 1.2834, Val Loss: 3.7718
Epoch 7: Loss = 1.2834, Prediction for y_pred → [2.1878142]
tensor([3.1700])
Train Loss: 1.2812, Val Loss: 3.7584
Epoch 8: Loss = 1.2812, Prediction for y_pred → [2.1870277]
tensor([3.1700])
Train Loss: 1.2790, Val Loss: 3.7431
Epoch 9: Loss = 1.2790, Prediction 

In [181]:
new_data = X_test[0]
new_model.eval()

with torch.no_grad():
    print(y_test[0])
    prediction = new_model(new_data)
    print(prediction)

tensor([18.3100])
tensor([16.6043])


In [184]:
new_model.eval()
with torch.no_grad():
    test_pred = new_model(X_test)
    test_loss = criterion(test_pred, y_test)
    print(f"MSE is equal to {test_loss} which is within acceptable limit")

MSE is equal to 2.898808002471924 which is within acceptable limit
